# 01. Dataset Extraction and Preparation

Extracts pairs from raw videos, using Frame 0 as the initial state. The pairs are sampled uniformly and balanced to prevent class bias.

In [2]:
%load_ext autoreload
%autoreload 2
    
import os
import cv2
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

# Import custom utilities
from utils import *
from dataset import PayloadDataset, get_transforms, denormalize

# Ensure output directories exist
create_output_dirs()

# Set random seed
torch.manual_seed(42)

In [3]:
# Extraction logic
FRAME_STRIDE = 10      # Extract every 10th frame
BUFFER_FRAMES = 30     # Skip 1 second before the loss event

def extract_frame(video_path, frame_idx, save_path):
    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    cap.release()
    if ret:
        cv2.imwrite(str(save_path), frame)
        return True
    return False

gt_df = pd.read_csv(GROUND_TRUTH_CSV)
pairs_data = []

print("Extracting frames...")

for index, row in gt_df.iterrows():
    vid_name = row['filename'] 
    is_loss = row['is_loss_event']
    loss_frame = row['loss_frame']
    
    vid_path = LOSS_VIDEOS / vid_name if is_loss else NORMAL_VIDEOS / vid_name
    if not vid_path.exists():
        continue
        
    cap = cv2.VideoCapture(str(vid_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    
    # Extract Initial Frame
    init_filename = f"{vid_name}_init.jpg"
    init_save_path = INIT_DIR / init_filename
    if not init_save_path.exists(): 
        extract_frame(vid_path, 0, init_save_path)

    # Uniform sampling for NORMAL videos
    if not is_loss:
        for f in range(30, total_frames - 1, FRAME_STRIDE):
            curr_filename = f"{vid_name}_curr_f{f}.jpg"
            if extract_frame(vid_path, f, CURR_DIR / curr_filename):
                pairs_data.append((init_save_path, CURR_DIR / curr_filename, 0))

    # Uniform sampling for LOSS videos
    else:
        # Pre-loss frames (Label 0)
        safe_upper_bound = max(1, loss_frame - BUFFER_FRAMES)
        if safe_upper_bound > 30:
            for f in range(30, safe_upper_bound, FRAME_STRIDE):
                curr_filename = f"{vid_name}_curr_f{f}.jpg"
                if extract_frame(vid_path, f, CURR_DIR / curr_filename):
                    pairs_data.append((init_save_path, CURR_DIR / curr_filename, 0))
        
        # Post-loss frames (Label 1)
        if total_frames > loss_frame + 5:
            for f in range(loss_frame, total_frames - 1, FRAME_STRIDE):
                curr_filename = f"{vid_name}_curr_f{f}.jpg"
                if extract_frame(vid_path, f, CURR_DIR / curr_filename):
                    pairs_data.append((init_save_path, CURR_DIR / curr_filename, 1))

print(f"Raw extraction complete. Total pairs: {len(pairs_data)}")

Extracting frames...


KeyError: 'video_name'

In [ ]:
# Balancing the Dataset
pairs_df = pd.DataFrame(pairs_data, columns=['initial_path', 'current_path', 'label'])

df_pos = pairs_df[pairs_df['label'] == 1]
df_neg = pairs_df[pairs_df['label'] == 0]

# Downsample the majority class
min_samples = min(len(df_pos), len(df_neg))
df_pos_sampled = df_pos.sample(n=min_samples, random_state=42)
df_neg_sampled = df_neg.sample(n=min_samples, random_state=42)

# Combine and shuffle
balanced_df = pd.concat([df_pos_sampled, df_neg_sampled]).sample(frac=1, random_state=42)
balanced_df.to_csv(PAIRS_CSV_PATH, index=False)

print(f"Balanced dataset saved. Total pairs: {len(balanced_df)}")
print(f"Label Distribution:\n{balanced_df['label'].value_counts()}")

In [ ]:
# 1. Create a temporary column to extract the base video name from the image path
balanced_df['video_name'] = balanced_df['initial_path'].apply(
    lambda x: os.path.basename(x).split('_init')[0]
)

# 2. Get the unique list of video names
unique_videos = balanced_df['video_name'].unique()

# 3. Split the list of videos: 70% Train, 30% Temp (which will become Val + Test)
train_vids, temp_vids = train_test_split(unique_videos, test_size=0.3, random_state=42)

# 4. Split the Temp videos: 50% Val, 50% Test (effectively 15% Val, 15% Test overall)
val_vids, test_vids = train_test_split(temp_vids, test_size=0.5, random_state=42)

# 5. Filter the image pairs based on which video they belong to
train_df = balanced_df[balanced_df['video_name'].isin(train_vids)]
val_df = balanced_df[balanced_df['video_name'].isin(val_vids)]
test_df = balanced_df[balanced_df['video_name'].isin(test_vids)]

# 6. Convert DataFrames back to the tuples expected by the PayloadDataset
train_pairs = list(train_df[['initial_path', 'current_path', 'label']].itertuples(index=False, name=None))
val_pairs = list(val_df[['initial_path', 'current_path', 'label']].itertuples(index=False, name=None))
test_pairs = list(test_df[['initial_path', 'current_path', 'label']].itertuples(index=False, name=None))

TARGET_RESOLUTION = (144, 256)
BATCH_SIZE = 16

# 7. Create Datasets (Train gets augmentations; Val and Test do NOT)
train_dataset = PayloadDataset(train_pairs, transform=get_transforms(is_train=True, use_augmentation=True, resolution=TARGET_RESOLUTION))
val_dataset = PayloadDataset(val_pairs, transform=get_transforms(is_train=False, use_augmentation=False, resolution=TARGET_RESOLUTION))
test_dataset = PayloadDataset(test_pairs, transform=get_transforms(is_train=False, use_augmentation=False, resolution=TARGET_RESOLUTION))

# 8. Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training pairs: {len(train_pairs)} | Batches: {len(train_loader)}")
print(f"Validation pairs: {len(val_pairs)} | Batches: {len(val_loader)}")
print(f"Testing pairs: {len(test_pairs)} | Batches: {len(test_loader)}")

In [ ]:
# Visualize Augmentations
def visualize_batch(dataloader, num_pairs=3):
    initial_imgs, current_imgs, labels = next(iter(dataloader))
    
    fig, axes = plt.subplots(num_pairs, 2, figsize=(10, 4 * num_pairs))
    
    for i in range(num_pairs):
        img1 = denormalize(initial_imgs[i]).permute(1, 2, 0).numpy()
        img2 = denormalize(current_imgs[i]).permute(1, 2, 0).numpy()
        
        img1 = torch.clamp(torch.tensor(img1), 0, 1).numpy()
        img2 = torch.clamp(torch.tensor(img2), 0, 1).numpy()
        
        lbl = int(labels[i].item())
        status = "LOSS (1)" if lbl == 1 else "PRESENT (0)"
        
        axes[i, 0].imshow(img1)
        axes[i, 0].set_title(f"Initial Anchor")
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(img2)
        axes[i, 1].set_title(f"Current State - {status}")
        axes[i, 1].axis('off')
        
    plt.tight_layout()
    plt.show()

visualize_batch(train_loader)